In [ ]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession

In [ ]:
points_schema = T.StructType([
    T.StructField("mission_id", T.StringType(), True),
    T.StructField("points", T.ArrayType(
        T.StructType([
            T.StructField("loc_id", T.StringType(), True),
            T.StructField("time", T.LongType(),True),
            T.StructField("latitude", T.DoubleType(),True),
            T.StructField("longitude", T.DoubleType(),True),
        ]),True),True)
    ])

In [ ]:
dataDF = [
    (('m0'),[('a',0,20.0,21.0),('b',1,30.0,31.0)]),
    (('m1'),[('a',None,20.0,21.0),('b',1,30.0,31.0)]),
    (('m2'),[('a',0,20.0,21.0),('b',1,None,31.0)]),
    (('m3'),[('a',0,20.0,None),('b',1,30.0,31.0)]),
]

In [ ]:
spark = SparkSession.builder.appName('blah').getOrCreate()

In [ ]:
tdf = spark.createDataFrame(data = dataDF, schema = points_schema)

In [ ]:
tdf.show(truncate=False)

In [ ]:
none_string = "*"
separator_string = ":"

def stringify_points(array_of_points:T.ArrayType(points_schema)) -> str:
    output = ""
    for x in array_of_points:
        output += (none_string if x.time is None else str(x.time)) + " "
        output += (none_string if x.latitude is None else f'{x.latitude:.6f}') + " "
        output += (none_string if x.longitude is None else f'{x.longitude:.6f}') + separator_string

    # drop the trailing tuple separator
    if len(array_of_points) > 0:
        output = output[0:len(separator_string) * -1]
    return output

In [ ]:
stringify_points_udf = F.udf(stringify_points, T.StringType())

In [ ]:
xdf = tdf.withColumn("stringified_points", stringify_points_udf("points"))

In [ ]:
xdf.show(truncate=False)